# LLM-MedQA: FastEmbed GPU Remote API

This notebook turns a cloud GPU instance (Kaggle/Colab) into a Remote Embedding API Server.
It uses FastAPI to serve the FastEmbed model, and `cloudflared` to expose it to the internet for free without needing an account.

**Instructions:**
1. Turn on the GPU (T4 x2 or P100 on Kaggle).
2. Run all cells.
3. Wait for the `YOUR PUBLIC API URL IS: https://...` line to print at the bottom.
4. On your local machine, run your pipeline with the `--remote-embed-url <URL>` flag.

In [ ]:
# 1. Install Dependencies
!pip install -q fastapi uvicorn pydantic nest-asyncio fastembed-gpu

In [ ]:
# 2. Download Cloudflare Tunnel daemon
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
from fastembed import TextEmbedding
import subprocess
import threading
import time
import re
import uvicorn
import nest_asyncio

# === APP CONFIG ===
MODEL_NAME = "BAAI/bge-small-en-v1.5"
PORT = 8000

app = FastAPI(title="LLM-MedQA FastEmbed Server")
embedder = None

@app.on_event("startup")
def load_model():
    global embedder
    print(f"\n>>> Loading model {MODEL_NAME} on GPU...")
    embedder = TextEmbedding(model_name=MODEL_NAME, threads=0)
    print(">>> Model loaded successfully!\n")

class EmbedRequest(BaseModel):
    texts: list[str]

@app.post("/embed")
def embed_batch(req: EmbedRequest):
    # embed returns a generator of numpy arrays
    embeddings = list(embedder.embed(req.texts))
    # Convert numpy arrays to lists for JSON serialization
    return {"embeddings": [emb.tolist() for emb in embeddings]}

@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_NAME}

# === RUN TUNNEL ===
def start_tunnel():
    print("\n>>> Starting Cloudflare Tunnel...")
    # Start cloudflared as a background process redirecting to local port
    cf_proc = subprocess.Popen(
        ['./cloudflared-linux-amd64', 'tunnel', '--url', f'http://127.0.0.1:{PORT}'],
        stdout=subprocess.PIPE, 
        stderr=subprocess.STDOUT, 
        text=True
    )
    
    url_found = False
    for line in iter(cf_proc.stdout.readline, ''):
        match = re.search(r'https://[-a-zA-Z0-9]+\.trycloudflare\.com', line)
        if match:
            url = match.group(0)
            print("\n" + "="*60)
            print(f"🚀 YOUR PUBLIC API URL IS: {url}")
            print(f"📌 PASS THIS TO YOUR PIPELINE: --remote-embed-url {url}")
            print("="*60 + "\n")
            url_found = True
            break
            
    if not url_found:
        print("⚠️ Failed to get Cloudflare URL. Check logs/internet connection.")

# Start tunnel in background thread so it doesn't block FastAPI
threading.Thread(target=start_tunnel, daemon=True).start()

# === RUN SERVER ===
nest_asyncio.apply()
# Wait a sec for the print lines to look nice
time.sleep(1)
uvicorn.run(app, host="127.0.0.1", port=PORT, log_level="info")
